# Signal Lab — Label Parameter Workbench

Interactively explore and tune labelling parameters on raw OHLCV datasets before committing them for model training.

**Typical workflow:**
1. Load a dataset in the **Configuration** cell
2. Edit the **Label Parameters** cell to tune the regime and labelling logic
3. Inspect the **Signal Chart** — buy/sell marker density, regime overlay, and slope signal
4. When happy, run **Save Parameters** to persist the config to `label_params.json`

| Stage | Description |
|---|---|
| Configuration | Choose instrument, timeframe, and evaluation window |
| Label Parameters | Tune regime detection, ATR multipliers, horizon, and pullback thresholds |
| Compute Labels | Run the causal triple-barrier labeller and trade-outcome calculator |
| Signal Chart | Interactive candlestick with regime-coloured MA, buy/sell markers, and slope panel |
| Save / Load | Persist or restore named parameter sets via `label_params.json` |

In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Learn.features import donchian_trend
warnings.filterwarnings('ignore')

PARAMS_FILE = 'label_params.json'   # saved in ModelWorkbench/

## Helpers

In [ ]:
def load_ohlcv(ds_name, start_date=None, n_rows=None):
    """Load an OHLCV CSV, optionally sliced by start_date or tail n_rows."""
    df = pd.read_csv(ds_name)
    df = df.sort_values('Time').reset_index(drop=True)
    df['Time'] = pd.to_datetime(df['Time'])
    if start_date:
        df = df[df['Time'] > start_date].reset_index(drop=True)
    if n_rows is not None:
        df = df.tail(n_rows).reset_index(drop=True)
    return df

## Configuration

Set the dataset path and evaluation window. `TEST_DATE` trims the dataframe to only post-date rows so the chart isn't overwhelmed — set to `None` to use the full file.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DS_NAME    = '../data/XAUUSD_1minute.csv'
TEST_DATE  = '2026-03-01'   # Show bars after this date only (None = full file)

# ── Load ──────────────────────────────────────────────────────────────────────
df = load_ohlcv(DS_NAME, start_date=TEST_DATE)

print(f"Dataset: {DS_NAME.split('/')[-1]}")
print(f"Window:  {df['Time'].iloc[0]}  →  {df['Time'].iloc[-1]}")
print(f"Bars:    {len(df):,}")

## Label Parameters

Tune the regime-detection and triple-barrier labelling parameters below, then run **Compute Labels** to see updated signals on the chart.

| Group | Key Parameters |
|---|---|
| **Regime** | `ma_period`, `slope_threshold`, `atr_percentile` — controls trend detection sensitivity |
| **Labelling** | `tp_mult`, `sl_mult`, `max_horizon` — risk-reward and time limit for each label |
| **Selectivity** | `trend_pullback_thresh` — raise to label fewer, higher-quality entries; `skip_range` masks range-bound bars |

Use the **Save / Load Parameters** cell below to persist a named config to `label_params.json`.

In [ ]:
# ── Instrument / Timeframe identifier (used as key in label_params.json) ──────
PARAM_KEY = 'XAUUSD_very_selective'

# ── Regime detection ─────────────────────────────────────────────────────────
regime_params = {
      "ma_period": 30,
      "slope_smoothness": 30,
      "regime_min_duration": 0,
      "atr_window": 60,
      "atr_lookback": 1440,
      "atr_percentile": 0.0,
      "slope_threshold": 0.1
    }

# ── Triple-barrier labelling ──────────────────────────────────────────────────
label_params = {
      "z_window": 14,
      "z_thresh": 1,
      "z_limit": 5,
      "atr_window": 14,
      "tp_mult": 5.0,
      "sl_mult": 2.5,
      "max_horizon": 90,
      "trend_pullback_thresh": 1.0,
      "regime_params": regime_params,
      "skip_range": True
    }

# ── Outcome params (subset used for P&L simulation) ──────────────────────────
outcome_params = {k: label_params[k] for k in ['atr_window', 'tp_mult', 'sl_mult', 'max_horizon']}

print(f"Active config:  '{PARAM_KEY}'")

## Save / Load Parameters

Run the **save** cell after tuning to persist the current config under `PARAM_KEY` in `label_params.json`.  
Run the **load** cell to restore a previously saved config (overwrites the params cells in memory — re-run Compute Labels afterwards).

In [ ]:
def _load_all_params(path=PARAMS_FILE):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}

def save_params(key=PARAM_KEY, path=PARAMS_FILE):
    """Persist current regime_params + label_params under `key` in the JSON file."""
    store = _load_all_params(path)
    store[key] = {
        'regime_params': regime_params,
        'label_params':  {k: v for k, v in label_params.items() if k != 'regime_params'},
    }
    with open(path, 'w') as f:
        json.dump(store, f, indent=2)
    print(f"Saved '{key}' → {path}  ({len(store)} entries total)")
    print("Keys:", list(store.keys()))

def load_params(key=PARAM_KEY, path=PARAMS_FILE):
    """Load a named config from the JSON file into the current session."""
    global regime_params, label_params, outcome_params
    store = _load_all_params(path)
    if key not in store:
        print(f"Key '{key}' not found in {path}. Available: {list(store.keys())}")
        return
    entry = store[key]
    regime_params = entry['regime_params']
    label_params  = {**entry['label_params'], 'regime_params': regime_params}
    outcome_params = {k: label_params[k] for k in ['atr_window', 'tp_mult', 'sl_mult', 'max_horizon']}
    print(f"Loaded '{key}' from {path}")
    return entry

# ── Uncomment the action you want ────────────────────────────────────────────
save_params()
# load_params()

## Compute Labels

Runs the causal triple-barrier labeller, the trade-outcome calculator, and the regime detector. Re-run this cell after changing any parameter above.

In [ ]:
from Learn.labels import (
    causal_triple_barrier_hilow_trend_labeler,
    causal_market_regime,
    super_smoother,
    calculate_trade_outcomes_all_candles,
)
from talib import EMA, ATR

# ── Labels ────────────────────────────────────────────────────────────────────
df_signals = causal_triple_barrier_hilow_trend_labeler(df.copy(), **label_params).rename(columns={'side': 'target'})
winning_trades = df_signals[df_signals['label'] == 1]
df['target'] = 0
df.loc[winning_trades.index, 'target'] = winning_trades['target']

counts = df['target'].value_counts()
n_buy  = counts.get( 1, 0)
n_sell = counts.get(-1, 0)
n_hold = counts.get( 0, 0)
print(f"Labels — BUY: {n_buy:,}  SELL: {n_sell:,}  HOLD: {n_hold:,}  "
      f"({(n_buy + n_sell) / len(df) * 100:.2f}% active)")

# ── Trade outcomes ────────────────────────────────────────────────────────────
df_outcomes = calculate_trade_outcomes_all_candles(df, **outcome_params)

# ── Regime & slope ────────────────────────────────────────────────────────────
df['Centered_MA']   = EMA(df['Close'], timeperiod=regime_params['ma_period'])
df['Regime']        = causal_market_regime(df, **regime_params)
slope               = df['Centered_MA'].diff()
df['slope_sm']      = super_smoother(slope, regime_params['slope_smoothness'])
df['donchain_trend'] = donchian_trend(df, length=regime_params['atr_window'])

regime_counts = df['Regime'].value_counts().rename({1: 'Up', 0: 'Range', -1: 'Down'})
print(f"Regime   — {dict(regime_counts)}")

## Signal Chart

Interactive candlestick chart showing:
- **▲ Green triangle (below bar)** — BUY label
- **▼ Red triangle (above bar)** — SELL label
- **Regime-coloured MA** — green (uptrend), red (downtrend), black (range)
- **Smoothed slope** and **Donchian trend** panels for regime diagnosis

In [ ]:
from talib import ATR

buy_bars  = df[df['target'] ==  1]
sell_bars = df[df['target'] == -1]

# Calculate ATR
df['ATR'] = ATR(df['High'], df['Low'], df['Close'], timeperiod=label_params['atr_window'])

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.02,
)

# ── Candlesticks ──────────────────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df['Time'],
    open=df['Open'], high=df['High'],
    low=df['Low'],   close=df['Close'],
    name='OHLCV',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
    increasing_fillcolor='#26a69a',  decreasing_fillcolor='#ef5350',
    line=dict(width=1),
    hoverinfo='skip',
), row=1, col=1)

# ── ATR Bands (Upper & Lower) ─────────────────────────────────────────────────
atr_mult = label_params['tp_mult']
df['Upper_Band'] = df['Close'] + df['ATR'] * atr_mult
df['Lower_Band'] = df['Close'] - df['ATR'] * atr_mult

fig.add_trace(go.Scatter(
    x=df['Time'], y=df['Upper_Band'],
    mode='lines',
    line=dict(width=1, color='rgba(0, 230, 118, 0.3)', dash='dot'),
    name=f'TP Upper ({atr_mult}×ATR)',
    hoverinfo='skip',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df['Time'], y=df['Lower_Band'],
    mode='lines',
    line=dict(width=1, color='rgba(255, 23, 68, 0.3)', dash='dot'),
    name=f'TP Lower ({atr_mult}×ATR)',
    hoverinfo='skip',
), row=1, col=1)

# ── BUY markers ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=buy_bars['Time'], y=buy_bars['Low'],
    mode='markers',
    marker=dict(symbol='triangle-up', size=14, color='#00e676',
                line=dict(color='darkgreen', width=0.8)),
    name='BUY label',
    hoverinfo='skip',
), row=1, col=1)

# ── SELL markers ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=sell_bars['Time'], y=sell_bars['High'],
    mode='markers',
    marker=dict(symbol='triangle-down', size=14, color='#ff1744',
                line=dict(color='darkred', width=0.8)),
    name='SELL label',
    hoverinfo='skip',
), row=1, col=1)

# ── Regime-coloured MA ────────────────────────────────────────────────────────
for regime_val, color, label in [(1, '#26a69a', 'Uptrend'), (0, '#888888', 'Range'), (-1, '#ef5350', 'Downtrend')]:
    fig.add_trace(go.Scatter(
        x=df['Time'],
        y=df['Centered_MA'].where(df['Regime'] == regime_val),
        mode='lines', line=dict(width=2, color=color), name=label,
        hoverinfo='skip',
    ), row=1, col=1)

# ── Slope & Donchian panels ───────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df['Time'], y=df['slope_sm'],
    mode='lines', line=dict(width=1, color='#82aaff'), name='Smoothed Slope',
    hoverinfo='skip',
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df['Time'], y=df['donchain_trend'],
    mode='lines', line=dict(width=1, color='#ffcb6b'), name='Donchian Trend',
    hoverinfo='skip',
), row=2, col=1)

# ── Layout ────────────────────────────────────────────────────────────────────
title_str = (
    f"{DS_NAME.split('/')[-1].replace('.csv','')}  |  "
    f"BUY: {len(buy_bars):,}   SELL: {len(sell_bars):,}   "
    f"({(len(buy_bars) + len(sell_bars)) / len(df) * 100:.2f}% active)  |  "
    f"Config: {PARAM_KEY}"
)

fig.update_layout(
    title=dict(text=title_str, font_size=13),
    xaxis_rangeslider_visible=False,
    xaxis2=dict(title='Time'),
    yaxis=dict(title='Price', fixedrange=False),
    yaxis2=dict(title='Indicator'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=1000,
    margin=dict(l=60, r=20, t=80, b=60),
    hovermode=False,
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    font=dict(color='#d4d4d4'),
    xaxis_gridcolor='#333',
    yaxis_gridcolor='#333',
    xaxis2_gridcolor='#333',
    yaxis2_gridcolor='#333',
)

fig.show()